# PRS Pipeline Benchmark

## Setup

In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr


MODE = "single"   # "single" or "multi"

AGENT_OUT = Path("/home/yj348/gibbs/AI_agent/PRS/Claudecode/single_ancestry")
GOLD_REF  = Path("/home/yj348/gibbs/AI_agent/PRS/ref")

# Benchmark trait set
TRAITS = ["BMI", "Height", "T2D"]

# For multi-ancestry only
ANCESTRIES = ["EUR", "AFR", "EAS", "SAS", "AMR"]

# Target genotype bfile prefix(es)
# Single: one prefix. Multi: per-ancestry dict {ancestry: bfile_prefix}.
TARGET_BFILE_SINGLE = Path("/gpfs/gibbs/pi/zhao/lx94/JointPRS/data/ukbb_data/geno_data/EUR")
TARGET_BFILE_MULTI  = {anc: Path(f"/gpfs/gibbs/pi/zhao/lx94/JointPRS/data/ukbb_data/geno_data/{anc}") for anc in ANCESTRIES}

# Phenotype/covariate
PHENO_TEMPLATE = "/gpfs/gibbs/pi/zhao/lx94/EEPRS/data/ukbb_pheno_data/ukbb_{trait}.tsv"
COVAR_PATH     = Path("/gpfs/gibbs/pi/zhao/lx94/JointPRS/data/ukbb_data/cov_data/agesex20PC.csv")

# Numeric tolerances
TOL_SNP_FRAC          = 0.005   # MCMC method SNP count tolerance: |delta|/N
TOL_BETA_PEARSON      = 0.99
TOL_SCORE_PEARSON     = 0.999
TOL_SCORE_MEAN_SD_FRAC= 1e-3    # |delta mean| < 1e-3 * SD
TOL_R2_AUC            = 0.01


In [ ]:
# Results container and check helper
results = []  # list of dicts: {task, check, status, detail}

def check(task, name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    results.append({"task": task, "check": name, "status": status, "detail": detail})
    icon = "✓" if condition else "✗"
    suffix = f" — {detail}" if detail else ""
    print(f"{icon} [Task {task}] {name}{suffix}")

def safe(fn):
    """Run a check function and convert exceptions to FAIL."""
    try:
        fn()
    except Exception as e:
        check(getattr(fn, '__task__', '?'), fn.__name__, False, f"exception: {e}")

# Helper for ancestry iteration
def per_ancestry():
    if MODE == "multi":
        return ANCESTRIES
    return [None]  # single ancestry: one pass without a label

# Helper to assemble a per-trait path that may or may not include an ancestry sub-dir
def trait_path(trait, *parts, ancestry=None):
    if ancestry:
        return AGENT_OUT / trait / ancestry / Path(*parts)
    return AGENT_OUT / trait / Path(*parts)


## Task 1 — Method selection and input validation

Checks the agent's method documentation file, that the declared paths
resolve, and (for multi) that the combination_strategy block exists.

In [ ]:
print("=" * 70)
print("TASK 1 — Method selection and input validation")
print("=" * 70)

# Locate the method documentation file (commonly method_card.json)
method_card_path = AGENT_OUT / "method_card.json"
exists = method_card_path.exists()
check(1, "Method documentation file exists", exists,
      detail=str(method_card_path) if exists else "missing")

if exists:
    try:
        mc = json.loads(method_card_path.read_text())
        check(1, "Method documentation file parses as JSON", True)
    except json.JSONDecodeError as e:
        check(1, "Method documentation file parses as JSON", False, str(e))
        mc = {}

    # Required fields
    required_fields = ["method", "input_format", "output_format"]
    if MODE == "multi":
        required_fields.append("combination_strategy")
    missing = [f for f in required_fields if f not in mc]
    check(1, "Method documentation has all required fields", not missing,
          detail=f"missing: {missing}" if missing else "")

    # Method in allowed set
    if MODE == "single":
        allowed = {"PRS-CS", "LDpred2", "SBayesR", "lassosum2", "PRScs", "LDpred2-auto"}
    else:
        allowed = {"PRS-CSx", "JointPRS", "SDPRX", "ME-Bayes SL", "MEBayes-SL"}
    method_name = mc.get("method", "")
    in_allowed = any(a.lower() in method_name.lower() for a in allowed)
    check(1, "Chosen method is in allowed set", in_allowed,
          detail=f"got '{method_name}', allowed {sorted(allowed)}")

    # All declared file paths in method_card resolve
    def collect_paths(obj):
        """Recursively collect string values that look like absolute file paths."""
        paths = []
        if isinstance(obj, dict):
            for v in obj.values():
                paths.extend(collect_paths(v))
        elif isinstance(obj, list):
            for v in obj:
                paths.extend(collect_paths(v))
        elif isinstance(obj, str) and obj.startswith("/"):
            paths.append(obj)
        return paths

    declared = collect_paths(mc)
    unresolved = [p for p in declared if not Path(p).exists()]
    check(1, "All declared file paths resolve",
          len(unresolved) == 0,
          detail=f"{len(unresolved)} unresolved" if unresolved else f"{len(declared)} paths OK")

# Sanity-check the trait manifest exists and is non-empty
manifest_path = Path("/gpfs/gibbs/pi/zhao/yj348/AI_agent/summary_stats_path.txt")
mf_ok = manifest_path.exists() and manifest_path.stat().st_size > 0
check(1, "Trait manifest exists and is non-empty", mf_ok, detail=str(manifest_path))

# Covariate and phenotype files exist for every trait
for trait in TRAITS:
    pheno = Path(PHENO_TEMPLATE.format(trait=trait))
    check(1, f"Phenotype file exists for {trait}", pheno.exists(), detail=str(pheno))
check(1, "Covariate file exists", COVAR_PATH.exists(), detail=str(COVAR_PATH))

# Target bfile triplet exists
if MODE == "single":
    bfiles = {"EUR": TARGET_BFILE_SINGLE}
else:
    bfiles = TARGET_BFILE_MULTI
for anc, prefix in bfiles.items():
    triplet_ok = all((Path(str(prefix) + ext)).exists() for ext in (".bed", ".bim", ".fam"))
    check(1, f"Target bfile triplet exists ({anc})", triplet_ok, detail=str(prefix))


## Task 2 — Data preprocessing and harmonization

Checks the cleaned, harmonized sumstats; per-ancestry medians; A1 alignment
to target .bim; and subsetted phenotype/covariate files.

In [ ]:
print()
print("=" * 70)
print("TASK 2 — Data preprocessing and harmonization")
print("=" * 70)

REQ_METHOD_COLS = None  # filled per-trait from method_card if available

for trait in TRAITS:
    for anc in per_ancestry():
        label = f"{trait}" + (f"/{anc}" if anc else "")

        # Locate cleaned sumstats
        clean_path = trait_path(trait, "sumstats_formatted.txt", ancestry=anc)
        harm_path  = trait_path(trait, "sumstats_harmonized.txt", ancestry=anc)
        target = harm_path if harm_path.exists() else clean_path
        if not target.exists():
            check(2, f"Cleaned/harmonized sumstats exists ({label})", False, str(target))
            continue
        check(2, f"Cleaned/harmonized sumstats exists ({label})", True)

        # Load with auto delimiter
        try:
            df = pd.read_csv(target, sep=None, engine='python')
        except Exception as e:
            check(2, f"Sumstats loads ({label})", False, str(e))
            continue

        # Zero duplicates (by SNP id)
        snp_col = "SNP" if "SNP" in df.columns else df.columns[0]
        n_dup = df[snp_col].duplicated().sum()
        check(2, f"Zero duplicate SNPs ({label})", n_dup == 0, detail=f"{n_dup} duplicates")

        # Zero non-positive SE
        if "SE" in df.columns:
            n_bad_se = (df["SE"] <= 0).sum() + df["SE"].isna().sum()
            check(2, f"Zero non-positive/NA SE ({label})", n_bad_se == 0, detail=f"{n_bad_se}")

        # Zero P out of (0, 1]
        if "P" in df.columns:
            n_bad_p = ((df["P"] <= 0) | (df["P"] > 1) | df["P"].isna()).sum()
            check(2, f"Zero P outside (0, 1] ({label})", n_bad_p == 0, detail=f"{n_bad_p}")

        # Zero non-finite BETA
        if "BETA" in df.columns:
            n_bad_beta = (~np.isfinite(df["BETA"])).sum()
            check(2, f"Zero non-finite BETA ({label})", n_bad_beta == 0, detail=f"{n_bad_beta}")

        # Median N matches independent recomputation
        if "N" in df.columns:
            implied_median = float(df["N"].median())
            qc_summary = trait_path(trait, "qc_summary.json", ancestry=anc)
            if qc_summary.exists():
                qs = json.loads(qc_summary.read_text())
                # qc_summary may have a top-level median_N or per-ancestry block
                reported = qs.get("median_N")
                if reported is None and anc and anc in qs:
                    reported = qs[anc].get("median_N")
                if reported is not None:
                    ok = abs(reported - implied_median) < 1.0  # integer-rounded N
                    check(2, f"Reported median N matches recomputation ({label})",
                          ok, detail=f"reported {reported} vs computed {implied_median}")

# Per-ancestry Ns are not all identical (multi only)
if MODE == "multi":
    for trait in TRAITS:
        ns = []
        qc_path = AGENT_OUT / trait / "qc_summary.json"
        if qc_path.exists():
            qs = json.loads(qc_path.read_text())
            for anc in ANCESTRIES:
                if anc in qs and "median_N" in qs[anc]:
                    ns.append(qs[anc]["median_N"])
        if len(ns) >= 2:
            distinct = len(set(ns)) > 1
            check(2, f"Per-ancestry Ns are not all identical ({trait})",
                  distinct, detail=f"Ns: {ns}")

# A1 in harmonized file matches target .bim A1 for 100% of retained SNPs
def load_bim(prefix):
    return pd.read_csv(str(prefix) + ".bim", sep=r'\s+', header=None,
                       names=["CHR", "SNP", "CM", "POS", "A1", "A2"])

for trait in TRAITS:
    for anc in per_ancestry():
        label = f"{trait}" + (f"/{anc}" if anc else "")
        harm_path = trait_path(trait, "sumstats_harmonized.txt", ancestry=anc)
        if not harm_path.exists():
            continue
        bim_prefix = TARGET_BFILE_MULTI[anc] if anc else TARGET_BFILE_SINGLE
        try:
            harm = pd.read_csv(harm_path, sep=None, engine='python')
            bim = load_bim(bim_prefix)
            merged = harm.merge(bim[["SNP", "A1"]], on="SNP", suffixes=("", "_bim"))
            if len(merged) == 0:
                check(2, f"A1 matches target .bim for retained SNPs ({label})",
                      False, "no overlap with .bim")
            else:
                match_rate = (merged["A1"] == merged["A1_bim"]).mean()
                check(2, f"A1 matches target .bim for retained SNPs ({label})",
                      match_rate >= 0.999,
                      detail=f"{match_rate*100:.2f}% match")
        except Exception as e:
            check(2, f"A1 matches target .bim for retained SNPs ({label})",
                  False, f"exception: {e}")

# Cross-ancestry A1 consistency (multi only)
if MODE == "multi":
    for trait in TRAITS:
        # Load each ancestry's harmonized sumstats
        frames = {}
        for anc in ANCESTRIES:
            p = trait_path(trait, "sumstats_harmonized.txt", ancestry=anc)
            if p.exists():
                frames[anc] = pd.read_csv(p, sep=None, engine='python')[["SNP", "A1"]]
        if len(frames) >= 2:
            # Intersect on SNP and check A1 identical
            common = None
            for anc, df in frames.items():
                df = df.rename(columns={"A1": f"A1_{anc}"})
                common = df if common is None else common.merge(df, on="SNP")
            if common is not None and len(common) > 0:
                a1_cols = [c for c in common.columns if c.startswith("A1_")]
                inconsistent = (common[a1_cols].nunique(axis=1) > 1).sum()
                check(2, f"Cross-ancestry A1 identical in intersection ({trait})",
                      inconsistent == 0,
                      detail=f"{inconsistent} inconsistent of {len(common)} shared SNPs")

# Subsetted phenotype/covariate files exist and have correct properties
for trait in TRAITS:
    for anc in per_ancestry():
        suffix = f"_{anc}" if anc else ""
        pheno_sub = AGENT_OUT / "phenotypes" / f"{trait}{suffix}_pheno_subset.tsv"
        covar_sub = AGENT_OUT / "phenotypes" / f"{trait}{suffix}_covar_subset.tsv"
        # Be tolerant: also accept variants without the suffix
        if not pheno_sub.exists():
            pheno_sub = AGENT_OUT / "phenotypes" / f"{trait}_pheno_subset.tsv"
        if not covar_sub.exists():
            covar_sub = AGENT_OUT / "phenotypes" / f"{trait}_covar_subset.tsv"
        label = f"{trait}" + (f"/{anc}" if anc else "")

        check(2, f"Phenotype subset exists ({label})", pheno_sub.exists(),
              detail=str(pheno_sub))
        check(2, f"Covariate subset exists ({label})", covar_sub.exists(),
              detail=str(covar_sub))

        if pheno_sub.exists() and covar_sub.exists():
            try:
                p = pd.read_csv(pheno_sub, sep='\t')
                c = pd.read_csv(covar_sub, sep='\t')
                # Tab delimiter implied by successful tab parse with > 1 column
                tab_ok = p.shape[1] >= 2 and c.shape[1] >= 2
                check(2, f"Subset files are tab-delimited ({label})", tab_ok)

                # No IID in subsets is absent from .fam
                bim_prefix = TARGET_BFILE_MULTI[anc] if anc else TARGET_BFILE_SINGLE
                fam = pd.read_csv(str(bim_prefix) + ".fam", sep=r'\s+', header=None,
                                  names=["FID", "IID", "PID", "MID", "SEX", "PHENO"])
                fam_iids = set(fam["IID"].astype(str))
                p_extra = set(p["IID"].astype(str)) - fam_iids if "IID" in p.columns else set()
                c_extra = set(c["IID"].astype(str)) - fam_iids if "IID" in c.columns else set()
                check(2, f"All subset IIDs are in .fam ({label})",
                      len(p_extra) == 0 and len(c_extra) == 0,
                      detail=f"pheno-extra={len(p_extra)}, covar-extra={len(c_extra)}")
            except Exception as e:
                check(2, f"Subset files readable ({label})", False, str(e))

# (Multi only) One subset file per ancestry, not a single combined
if MODE == "multi":
    for trait in TRAITS:
        per_anc_files = []
        for anc in ANCESTRIES:
            p = AGENT_OUT / "phenotypes" / f"{trait}_{anc}_pheno_subset.tsv"
            if p.exists():
                per_anc_files.append(p)
        check(2, f"Per-ancestry subset files present ({trait})",
              len(per_anc_files) >= 2,
              detail=f"{len(per_anc_files)} ancestry-specific files")


## Task 3 — PRS execution

Checks per-chromosome weight files, merged weights, PLINK2 scoring command,
and .sscore outputs. Compares posterior BETAs and PRS scores against the
gold-standard reference.

In [ ]:
print()
print("=" * 70)
print("TASK 3 — PRS execution")
print("=" * 70)

def read_weights(path):
    return pd.read_csv(path, sep=None, engine='python')

for trait in TRAITS:
    for anc in per_ancestry():
        label = f"{trait}" + (f"/{anc}" if anc else "")
        weights_dir = trait_path(trait, "weights", ancestry=anc)

        # All 22 per-chromosome outputs exist
        if weights_dir.exists():
            chr_files = sorted(weights_dir.glob("chr*.txt"))
            present_chrs = set()
            for f in chr_files:
                m = re.search(r"chr(\d+)", f.name)
                if m:
                    present_chrs.add(int(m.group(1)))
            missing = sorted(set(range(1, 23)) - present_chrs)
            check(3, f"All 22 per-chromosome weights exist ({label})",
                  len(missing) == 0, detail=f"missing chr: {missing}" if missing else "")
        else:
            check(3, f"All 22 per-chromosome weights exist ({label})",
                  False, f"weights dir missing: {weights_dir}")

        # Merged weight file exists, has > 0 variants, no duplicates
        merged_path = trait_path(trait, "weights_merged.txt", ancestry=anc)
        if not merged_path.exists():
            check(3, f"Merged weight file exists ({label})", False, str(merged_path))
            continue
        check(3, f"Merged weight file exists ({label})", True)

        try:
            mw = read_weights(merged_path)
            snp_col = "SNP" if "SNP" in mw.columns else mw.columns[0]
            check(3, f"Merged weight file has >0 variants ({label})",
                  len(mw) > 0, detail=f"{len(mw)} variants")
            n_dup = mw[snp_col].duplicated().sum()
            check(3, f"Zero duplicate SNPs in merged weights ({label})",
                  n_dup == 0, detail=f"{n_dup} duplicates")
        except Exception as e:
            check(3, f"Merged weight file readable ({label})", False, str(e))
            continue

        # Pearson r against gold-standard posterior BETAs
        gold_path = (GOLD_REF / trait / (f"{anc}/" if anc else "") /
                     "weights_merged.txt") if anc else GOLD_REF / trait / "weights_merged.txt"
        gold_path = Path(str(gold_path).replace("//", "/"))
        if gold_path.exists():
            try:
                gw = read_weights(gold_path)
                beta_col_mw = next((c for c in mw.columns if "BETA" in c.upper()), None)
                beta_col_gw = next((c for c in gw.columns if "BETA" in c.upper()), None)
                if beta_col_mw and beta_col_gw:
                    merged = mw[[snp_col, beta_col_mw]].merge(
                        gw[[snp_col, beta_col_gw]], on=snp_col)
                    if len(merged) >= 100:
                        r, _ = pearsonr(merged[beta_col_mw], merged[beta_col_gw])
                        check(3, f"Posterior BETA Pearson r > {TOL_BETA_PEARSON} ({label})",
                              r > TOL_BETA_PEARSON, detail=f"r = {r:.4f}, n = {len(merged)}")
                    else:
                        check(3, f"Posterior BETA Pearson r > {TOL_BETA_PEARSON} ({label})",
                              False, f"only {len(merged)} overlapping SNPs with reference")
            except Exception as e:
                check(3, f"Posterior BETA comparison ({label})", False, str(e))

        # .sscore output checks
        sscore = trait_path(trait, "prs_scores.sscore", ancestry=anc)
        if not sscore.exists():
            # Multi may also place per-ancestry sscore at trait/{anc}/prs_scores.sscore
            check(3, f".sscore output exists ({label})", False, str(sscore))
            continue
        check(3, f".sscore output exists ({label})", True)

        try:
            s = pd.read_csv(sscore, sep=r'\s+')
            # Find the score column
            score_col = next((c for c in s.columns if c.startswith("SCORE")), None)
            bim_prefix = TARGET_BFILE_MULTI[anc] if anc else TARGET_BFILE_SINGLE
            fam = pd.read_csv(str(bim_prefix) + ".fam", sep=r'\s+', header=None,
                              names=["FID", "IID", "PID", "MID", "SEX", "PHENO"])
            # One row per .fam individual
            check(3, f".sscore row count == .fam row count ({label})",
                  len(s) == len(fam), detail=f"sscore={len(s)}, fam={len(fam)}")
            # Zero NA scores
            if score_col:
                n_na = s[score_col].isna().sum()
                check(3, f"Zero NA scores ({label})", n_na == 0, detail=f"{n_na} NA")

                # Mean and SD vs gold standard
                gold_sscore = (GOLD_REF / trait / (f"{anc}/" if anc else "")).resolve() / "prs_scores.sscore"
                if gold_sscore.exists():
                    gs = pd.read_csv(gold_sscore, sep=r'\s+')
                    gs_score = next((c for c in gs.columns if c.startswith("SCORE")), None)
                    if gs_score:
                        sd_gold = gs[gs_score].std()
                        mean_delta = abs(s[score_col].mean() - gs[gs_score].mean())
                        check(3, f"Score mean within {TOL_SCORE_MEAN_SD_FRAC}*SD of gold ({label})",
                              mean_delta < TOL_SCORE_MEAN_SD_FRAC * sd_gold,
                              detail=f"|delta mean|={mean_delta:.4g}, SD={sd_gold:.4g}")
                        merged_sc = s.merge(gs, on=["FID", "IID"], suffixes=("_a", "_g"))
                        if len(merged_sc) >= 100:
                            ra, _ = pearsonr(merged_sc[score_col + "_a"],
                                             merged_sc[gs_score + "_g"])
                            check(3, f"PRS Pearson r > {TOL_SCORE_PEARSON} ({label})",
                                  ra > TOL_SCORE_PEARSON, detail=f"r = {ra:.5f}")
        except Exception as e:
            check(3, f".sscore parseable ({label})", False, str(e))

# Inspect logged scoring command for --score + cols=+scoresums
for trait in TRAITS:
    for anc in per_ancestry():
        label = f"{trait}" + (f"/{anc}" if anc else "")
        cmd_path = trait_path(trait, "scoring_summary.json", ancestry=anc)
        if not cmd_path.exists():
            # Fall back to a shell script with the command
            cmd_path = trait_path(trait, "step5_score_command.sh", ancestry=anc)
        if cmd_path.exists():
            txt = cmd_path.read_text()
            has_score = "--score" in txt
            has_sum   = ("cols=+scoresums" in txt) or ("--score-sum" in txt) or ("--sum" in txt)
            check(3, f"PLINK2 command uses --score with sum option ({label})",
                  has_score and has_sum,
                  detail=f"--score={has_score}, sum={has_sum}")

# Multi-only: per-ancestry --score commands use matching bfile + weights
if MODE == "multi":
    for trait in TRAITS:
        for anc in ANCESTRIES:
            cmd_path = trait_path(trait, "scoring_summary.json", ancestry=anc)
            if cmd_path.exists():
                txt = cmd_path.read_text()
                expected_bfile = str(TARGET_BFILE_MULTI[anc])
                expected_weight = f"/{anc}/weights_merged.txt"
                bfile_ok = expected_bfile in txt
                weight_ok = expected_weight in txt
                check(3, f"Per-ancestry --score matches its bfile and weights ({trait}/{anc})",
                      bfile_ok and weight_ok,
                      detail=f"bfile_in_cmd={bfile_ok}, weight_path_in_cmd={weight_ok}")

# Random seed is logged somewhere in the method log
for trait in TRAITS:
    log_path = AGENT_OUT / trait / "logs"
    if log_path.exists():
        seed_found = False
        for f in log_path.glob("method_chr*.log"):
            if re.search(r"(seed|random[\s_-]?seed)", f.read_text(), re.IGNORECASE):
                seed_found = True
                break
        check(3, f"Random seed logged ({trait})", seed_found)


## Task 4 — SLURM execution

Checks the three-stage SLURM scripts, timing/memory logs, failure-classification
logic, and the pipeline summary report.

In [ ]:
print()
print("=" * 70)
print("TASK 4 — SLURM execution")
print("=" * 70)

scripts_dir = AGENT_OUT / "scripts"

# Required SBATCH headers per stage
expected_headers = {
    "stage1_preprocess.sh": {"--time=2:00:00", "--cpus-per-task=2", "--mem=16G",
                              "--partition=scavenge"},
    "stage2_weights.sh":    {"--time=24:00:00", "--cpus-per-task=4", "--mem=50G",
                              "--partition=scavenge"},
    "stage3_score.sh":      {"--time=4:00:00", "--cpus-per-task=4", "--mem=32G",
                              "--partition=scavenge"},
}

for script_name, required in expected_headers.items():
    p = scripts_dir / script_name
    if not p.exists():
        check(4, f"Stage script exists ({script_name})", False, str(p))
        continue
    check(4, f"Stage script exists ({script_name})", True)
    body = p.read_text()
    missing = [h for h in required if h not in body]
    check(4, f"SBATCH headers correct ({script_name})",
          len(missing) == 0, detail=f"missing: {missing}" if missing else "")

    # set -euo pipefail and trap
    has_setpipefail = "set -euo pipefail" in body
    has_trap = re.search(r"\btrap\b", body) is not None
    check(4, f"set -euo pipefail present ({script_name})", has_setpipefail)
    check(4, f"trap clause present ({script_name})", has_trap)

    # conda activate
    has_conda = ("conda activate" in body) or ("source activate" in body)
    check(4, f"conda activate in script ({script_name})", has_conda)

# Master run_prs.sh: sbatch --test-only parses; prints three job IDs
master = scripts_dir / "run_prs.sh"
if master.exists():
    check(4, "Master run_prs.sh exists", True)
    body = master.read_text()
    has_afterok = "afterok" in body
    check(4, "Master script chains stages with afterok", has_afterok)
    # Three sbatch submissions
    n_sbatch = len(re.findall(r"\bsbatch\b", body))
    check(4, "Master script issues 3 sbatch submissions", n_sbatch >= 3,
          detail=f"{n_sbatch} found")
    # Optional dry-run check via subprocess (skipped if sbatch unavailable)
    try:
        r = subprocess.run(["sbatch", "--test-only", str(master)],
                           capture_output=True, text=True, timeout=20)
        check(4, "sbatch --test-only on master script parses",
              r.returncode in (0, 1),  # --test-only often returns 1
              detail=f"rc={r.returncode}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        check(4, "sbatch --test-only on master script parses",
              True, "sbatch not available in this env — skipped")
else:
    check(4, "Master run_prs.sh exists", False, str(master))

# Stage 2 array mapping (static check): script computes trait_index and chrom from SLURM_ARRAY_TASK_ID
stage2 = scripts_dir / "stage2_weights.sh"
if stage2.exists():
    body = stage2.read_text()
    has_mapping = (re.search(r"SLURM_ARRAY_TASK_ID", body) is not None and
                   ("22" in body) and
                   (re.search(r"%\s*22|/\s*22", body) is not None))
    check(4, "Stage 2 array index → (trait, chrom) mapping present",
          has_mapping)

# Multi-only: Stage 2 joint invocation (all ancestries in one command)
if MODE == "multi" and stage2.exists():
    body = stage2.read_text()
    # Look for evidence of looping over ancestries to BUILD args (acceptable),
    # but NOT invoking the method once per ancestry separately.
    # Heuristic: the method invocation line should reference all ancestry inputs at once.
    has_joint_pattern = ("--sst_file" in body and "," in body) or \
                        ("--pop" in body) or \
                        ("ancestry_list" in body)
    check(4, "Stage 2 invokes joint multi-ancestry call (heuristic)",
          has_joint_pattern,
          detail="based on presence of multi-ancestry-aware CLI patterns")

# Timing logs exist with required fields
required_timing_fields = {"STAGE", "TRAIT", "STEP", "START", "END",
                          "ELAPSED_SEC", "TOTAL_ELAPSED_SEC"}
for trait in TRAITS:
    log_dir = AGENT_OUT / trait / "logs"
    if not log_dir.exists():
        check(4, f"Logs directory exists ({trait})", False, str(log_dir))
        continue
    timing_files = list(log_dir.glob("timing_*.log"))
    check(4, f"timing_*.log files present ({trait})", len(timing_files) > 0,
          detail=f"{len(timing_files)} files")
    if timing_files:
        sample = timing_files[0].read_text()
        present = {f for f in required_timing_fields if f in sample}
        check(4, f"timing logs have required fields ({trait})",
              len(present) >= len(required_timing_fields) - 1,
              detail=f"present: {sorted(present)}")

    # Memory logs
    mem_files = list(log_dir.glob("memory_*.log"))
    check(4, f"memory_*.log files present ({trait})", len(mem_files) > 0)

# Failure classification: parse any failure_*.log and check LIKELY_CAUSE is one of the known classes
KNOWN_CAUSES = {"OOM", "TIMEOUT", "FILE_NOT_FOUND", "INPUT_FORMAT_ERROR",
                "ZERO_OVERLAP", "GROUP_MISMATCH", "SOFTWARE_ERROR", "UNKNOWN"}
all_failure_logs = []
for trait in TRAITS:
    log_dir = AGENT_OUT / trait / "logs"
    if log_dir.exists():
        all_failure_logs.extend(log_dir.glob("failure_*.log"))
if all_failure_logs:
    bad = []
    for f in all_failure_logs:
        m = re.search(r"LIKELY_CAUSE:\s*(\S+)", f.read_text())
        if not m or m.group(1) not in KNOWN_CAUSES:
            bad.append(f.name)
    check(4, "All failure logs use a known LIKELY_CAUSE", len(bad) == 0,
          detail=f"{len(bad)} non-conforming" if bad else f"{len(all_failure_logs)} OK")

# pipeline_summary.tsv
psum = AGENT_OUT / "pipeline_summary.tsv"
if psum.exists():
    check(4, "pipeline_summary.tsv exists", True)
    try:
        df = pd.read_csv(psum, sep='\t')
        required_cols = {"TRAIT", "STAGE", "CHROM", "STATUS", "EXIT_CODE",
                         "ELAPSED_SEC", "PEAK_MEMORY_MB", "LIKELY_CAUSE"}
        if MODE == "multi":
            required_cols.add("GROUP")
        missing = required_cols - set(df.columns)
        check(4, "pipeline_summary.tsv has required columns", len(missing) == 0,
              detail=f"missing: {missing}" if missing else "")
    except Exception as e:
        check(4, "pipeline_summary.tsv parses", False, str(e))
else:
    check(4, "pipeline_summary.tsv exists", False, str(psum))

# generate_report.sh runs standalone
gen_report = scripts_dir / "generate_report.sh"
if gen_report.exists():
    check(4, "generate_report.sh exists", True)
    try:
        r = subprocess.run(["bash", str(gen_report)], capture_output=True,
                           text=True, timeout=120, cwd=str(AGENT_OUT))
        check(4, "generate_report.sh runs without error",
              r.returncode == 0, detail=f"rc={r.returncode}")
    except Exception as e:
        check(4, "generate_report.sh runs without error", False, str(e))
else:
    check(4, "generate_report.sh exists", False)


## Task 5 — PRS evaluation

Checks per-trait evaluation outputs, sample sizes, PRS standardization,
and incremental-R²/AUC consistency. For multi-ancestry, also checks the
per-(trait, ancestry) breakdown in `evaluation_summary.tsv`.

In [ ]:
print()
print("=" * 70)
print("TASK 5 — PRS evaluation")
print("=" * 70)

# Per-trait evaluation.json
for trait in TRAITS:
    eval_path = AGENT_OUT / trait / "evaluation.json"
    if not eval_path.exists():
        check(5, f"evaluation.json exists ({trait})", False, str(eval_path))
        continue
    check(5, f"evaluation.json exists ({trait})", True)

    try:
        ev = json.loads(eval_path.read_text())
    except Exception as e:
        check(5, f"evaluation.json parses ({trait})", False, str(e))
        continue

    # Multi-ancestry results are nested under 'per_ancestry'
    blocks = {None: ev} if MODE == "single" else ev.get("per_ancestry", {})

    for anc, block in blocks.items():
        label = f"{trait}" + (f"/{anc}" if anc else "")
        # Required fields
        required = {"N_individuals", "beta_PRS", "SE_PRS", "p_PRS"}
        # Either INCR_R2 or (AUC and NAGELKERKE_R2) must be present
        has_r2  = "INCR_R2" in block or "r2_incremental" in block
        has_auc = "AUC" in block or "auc_incremental" in block
        has_metric = has_r2 or has_auc
        missing = required - set(block.keys())
        # Be tolerant about case
        missing = {f for f in missing if f.lower() not in {k.lower() for k in block.keys()}}
        check(5, f"Evaluation has required fields ({label})",
              len(missing) == 0 and has_metric,
              detail=f"missing: {missing}; metric_present: {has_metric}")

        # Incremental R^2 consistency
        r2_full = block.get("r2_full") or block.get("R2_full")
        r2_base = block.get("r2_covariates_only") or block.get("BASELINE_R2_OR_AUC")
        r2_inc  = block.get("r2_incremental") or block.get("INCR_R2")
        if all(x is not None for x in [r2_full, r2_base, r2_inc]):
            delta = abs(r2_inc - (r2_full - r2_base))
            check(5, f"INCR_R2 == R2_full - R2_baseline ({label})",
                  delta < 1e-6, detail=f"delta = {delta:.2e}")

# Standardization check via reading the merged-frame side artifact, if present
for trait in TRAITS:
    for anc in per_ancestry():
        label = f"{trait}" + (f"/{anc}" if anc else "")
        merged_path = AGENT_OUT / trait / ("evaluation_merged.tsv" if not anc
                                           else f"{anc}/evaluation_merged.tsv")
        if merged_path.exists():
            try:
                m = pd.read_csv(merged_path, sep='\t')
                prs_col = next((c for c in m.columns if "PRS" in c.upper()
                                and ("STD" in c.upper() or "Z" in c.upper())), None)
                if prs_col is None:
                    prs_col = next((c for c in m.columns if c.upper().startswith("PRS")), None)
                if prs_col:
                    mu = m[prs_col].mean()
                    sd = m[prs_col].std()
                    standardized = abs(mu) < 0.05 and abs(sd - 1.0) < 0.05
                    check(5, f"PRS z-standardized in eval frame ({label})",
                          standardized,
                          detail=f"mean={mu:.3f}, sd={sd:.3f}")
            except Exception as e:
                pass  # silent — this artifact is optional

# evaluation_summary.tsv at root
esum = AGENT_OUT / "evaluation_summary.tsv"
if not esum.exists():
    check(5, "evaluation_summary.tsv exists at working dir root",
          False, str(esum))
else:
    check(5, "evaluation_summary.tsv exists at working dir root", True)
    try:
        df = pd.read_csv(esum, sep='\t')
        required = {"TRAIT", "PHENO_TYPE", "N_INDIVIDUALS", "BETA_PRS", "SE_PRS",
                    "P_PRS", "INCR_R2", "AUC", "NAGELKERKE_R2", "BASELINE_R2_OR_AUC"}
        if MODE == "multi":
            required.add("ANCESTRY")
        missing = required - set(df.columns)
        check(5, "evaluation_summary.tsv has required columns",
              len(missing) == 0, detail=f"missing: {missing}" if missing else "")
        if MODE == "multi" and "ANCESTRY" in df.columns:
            # One row per (trait, ancestry)
            n_rows = len(df)
            n_unique = df[["TRAIT", "ANCESTRY"]].drop_duplicates().shape[0]
            check(5, "One row per (trait, ancestry) in summary",
                  n_rows == n_unique, detail=f"{n_rows} rows, {n_unique} unique")
    except Exception as e:
        check(5, "evaluation_summary.tsv parses", False, str(e))

# Regression type consistency: linear for continuous, logistic for binary
# We infer the phenotype type from PHENO_TYPE in evaluation.json (or summary tsv)
for trait in TRAITS:
    eval_path = AGENT_OUT / trait / "evaluation.json"
    if not eval_path.exists():
        continue
    try:
        ev = json.loads(eval_path.read_text())
        # Look for pheno_type and either auc_incremental for binary or incr_r2 for continuous
        ptype = (ev.get("pheno_type") or
                 ev.get("PHENO_TYPE") or "").lower()
        # Take from per_ancestry if multi
        if not ptype and "per_ancestry" in ev:
            first = next(iter(ev["per_ancestry"].values()), {})
            ptype = (first.get("pheno_type") or first.get("PHENO_TYPE") or "").lower()
        if ptype:
            valid = (ptype == "continuous" or ptype == "binary")
            check(5, f"PHENO_TYPE is recognized ({trait})", valid,
                  detail=f"got '{ptype}'")
    except Exception:
        pass


## Summary

Aggregates check results per task and overall.

In [ ]:
print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

import collections
by_task = collections.defaultdict(lambda: {"pass": 0, "fail": 0})
for r in results:
    by_task[r["task"]][r["status"].lower()] += 1

total_pass = 0
total_fail = 0
print(f"{'Task':<6} {'Pass':>6} {'Fail':>6} {'Total':>6} {'Rate':>8}")
print("-" * 38)
for t in sorted(by_task):
    p = by_task[t]["pass"]
    f = by_task[t]["fail"]
    total = p + f
    rate = (p / total * 100) if total else 0
    print(f"Task {t:<2} {p:>6} {f:>6} {total:>6} {rate:>7.1f}%")
    total_pass += p
    total_fail += f
total = total_pass + total_fail
overall_rate = (total_pass / total * 100) if total else 0
print("-" * 38)
print(f"{'TOTAL':<6} {total_pass:>6} {total_fail:>6} {total:>6} {overall_rate:>7.1f}%")

# Write detailed results to a TSV for downstream analysis
results_df = pd.DataFrame(results)
out_csv = AGENT_OUT / "verification_results.tsv"
try:
    results_df.to_csv(out_csv, sep='\t', index=False)
    print(f"\nDetailed results written to: {out_csv}")
except Exception as e:
    print(f"\nCould not write results: {e}")

# List failed checks for quick triage
print("\nFailed checks (first 30):")
fails = [r for r in results if r["status"] == "FAIL"]
for r in fails[:30]:
    detail = f" — {r['detail']}" if r['detail'] else ""
    print(f"  Task {r['task']}: {r['check']}{detail}")
if len(fails) > 30:
    print(f"  ... and {len(fails) - 30} more")
